# 🏷️ Lab W4-2 — Metric Sprawl และ Semantic Layer

**รายวิชาระบบสนับสนุนการตัดสินใจ · สัปดาห์ที่ 4 — OLAP and Multidimensional Analysis**

Lab นี้ใช้คู่กับสื่อจำลอง **Metric Sprawl Arena** (`/sims/metric-sprawl`)
ตัวเลขที่คุณคำนวณได้ในสมุดเล่มนี้ต้องตรงกับตัวเลขบนหน้าจอสื่อจำลองทุกหลัก

## สิ่งที่จะได้เรียนรู้
1. อธิบายได้ว่า **metric sprawl** เกิดขึ้นได้อย่างไรทั้งที่ทุกฝ่ายใช้ข้อมูลชุดเดียวกัน
2. แยก **นิยามของตัววัด** ออกจาก **ข้อมูลดิบ** และจาก **เครื่องมือที่ใช้แสดงผล**
3. เขียนนิยามตัววัดแบบ **metrics-as-code** ที่ตรวจสอบและทดสอบได้
4. อธิบายว่า **semantic layer** แก้ปัญหาอะไร และแก้ไม่ได้อะไร

## ข้อมูล
`revenue_source.csv` — คำสั่งซื้อปี 2025 จำนวน 9,635 รายการ
มีคอลัมน์ครบทุกองค์ประกอบที่แต่ละฝ่ายเลือกหยิบไปคนละชุด

In [ ]:
import pandas as pd

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

URL = ("https://raw.githubusercontent.com/babankbro/ksu-dss-course/"
       "master/datasets/week04/revenue_source.csv")
df = pd.read_csv(URL)

print(f"จำนวนคำสั่งซื้อ : {len(df):,}")
print(f"ช่วงวันที่      : {df.order_date.min()} ถึง {df.order_date.max()}")
print(f"ช่องทาง         : {df.channel.value_counts().to_dict()}")
df.head(5)

## ส่วนที่ 1 — ที่ประชุมผู้บริหารเมื่อเช้านี้

คำถามเดียว: **"รายได้ปี 2025 เท่าไร"**
ทั้ง 4 ฝ่ายเปิดรายงานของตัวเอง แล้วได้ตัวเลขไม่ตรงกันสักฝ่าย

ก่อนอื่นให้ดูว่าในไฟล์เดียวกันนี้มี "ก้อนเงิน" ให้เลือกหยิบกี่ก้อน

In [ ]:
COMPONENTS = ["gross_amount", "discount", "net_amount",
              "vat_amount", "shipping_fee", "returned_amount"]

totals = df[COMPONENTS].sum()
print("ผลรวมของแต่ละองค์ประกอบ (บาท)")
print(totals.to_string())

print(f"\nตรวจความสอดคล้อง: net = gross − discount ?  "
      f"{bool((df.net_amount.round(2) == (df.gross_amount - df.discount).round(2)).all())}")
print(f"ตรวจ VAT: vat = net × 7/107 ?  ต่างสูงสุด {abs(df.vat_amount - df.net_amount*7/107).max():.4f} บาท")

> **ไม่มีคอลัมน์ใดผิด** ทุกตัวเลขคำนวณถูกต้องตามสูตรของตัวเอง
> ปัญหาอยู่ที่ **ไม่มีใครตกลงกันว่าคำว่า "รายได้" หมายถึงก้อนไหนบ้าง**

### 🧑‍💻 งานที่ 1
เขียนฟังก์ชัน `revenue(base, deduct)` ที่คำนวณรายได้ตามนิยามที่ส่งเข้ามา

* `base` — `"gross"` หรือ `"net"`
* `deduct` — เซ็ตของสิ่งที่ต้องการหัก จาก `{"returns", "vat", "shipping"}`

แล้วใช้ฟังก์ชันนี้คำนวณตัวเลขของทั้ง 4 ฝ่าย

| ฝ่าย | ฐาน | หัก |
|---|---|---|
| ฝ่ายขาย | gross | — |
| ฝ่ายการตลาด | net | คืนสินค้า |
| ฝ่ายบัญชี | net | คืนสินค้า, VAT |
| ผู้บริหาร | net | คืนสินค้า, VAT, ค่าขนส่ง |

*เฉลยที่ถูกต้อง: 13,797,768.00 / 12,610,979.50 / 11,773,005.38 / 11,725,165.38*

In [ ]:
# เขียนโค้ดของคุณที่นี่


> **ห่างกัน 17.68% จากข้อมูลชุดเดียวกัน** — ไม่ใช่ปัญหาคุณภาพข้อมูล
> ไม่ใช่ปัญหา ETL ไม่ใช่ปัญหาเครื่องมือ BI
>
> เป็นปัญหา **การกำกับดูแลนิยาม (metric governance)** ล้วน ๆ

## ส่วนที่ 2 — แต่ละฝ่ายไม่ได้ผิด

จุดที่นักศึกษามักเข้าใจผิดคือคิดว่า "ต้องมีฝ่ายใดฝ่ายหนึ่งผิด"
ความจริงคือแต่ละนิยามถูกต้องสำหรับ **คำถามของฝ่ายนั้น**

### 🧑‍💻 งานที่ 2
จับคู่แต่ละนิยามกับ **คำถามที่มันตอบได้ถูกต้อง** และ **คำถามที่มันตอบผิด**

เขียนเป็นตาราง 4 แถว มีคอลัมน์: ฝ่าย · ใช้ตัดสินใจเรื่องอะไร · ห้ามใช้ตอบคำถามอะไร

แนวคิด: VAT เป็นเงินที่เก็บแทนรัฐ ไม่ใช่รายได้ของบริษัท ·
ค่าขนส่งที่บริษัทออกให้เป็นต้นทุน ไม่ใช่ตัวหักรายได้ในงบการเงิน

In [ ]:
# เขียนโค้ดของคุณที่นี่


## ส่วนที่ 3 — Semantic layer: นิยามเป็นโค้ด

แทนที่จะให้แต่ละฝ่ายเขียน SQL ของตัวเอง
ให้นิยามตัววัดไว้ที่เดียวในรูปแบบที่เครื่องอ่านได้ แล้วทุกเครื่องมือดึงไปใช้

In [ ]:
METRIC_STORE = {
    "gross_revenue": {
        "label": "รายได้ขั้นต้น",
        "expr": lambda d: d.gross_amount.sum(),
        "owner": "ฝ่ายขาย",
        "use_for": "คอมมิชชันและ KPI ทีมขาย",
        "not_for": "งบการเงิน",
    },
    "net_revenue_recognized": {
        "label": "รายได้ที่รับรู้ทางบัญชี",
        "expr": lambda d: d.net_amount.sum() - d.returned_amount.sum() - d.vat_amount.sum(),
        "owner": "ฝ่ายบัญชี",
        "use_for": "งบการเงินและการยื่นภาษี",
        "not_for": "การวัดผลงานทีมขาย",
    },
    "cash_contribution": {
        "label": "เงินที่เหลือเข้าบริษัท",
        "expr": lambda d: (d.net_amount.sum() - d.returned_amount.sum()
                           - d.vat_amount.sum() - d.shipping_fee.sum()),
        "owner": "ฝ่ายการเงิน",
        "use_for": "ประเมินกระแสเงินสด",
        "not_for": "งบกำไรขาดทุน",
    },
}

print("ทะเบียนตัววัดกลาง (metric store)\n")
for name, m in METRIC_STORE.items():
    print(f"  {name:<24} {m['label']:<24} {m['expr'](df):>16,.2f}  · เจ้าของ: {m['owner']}")

### 🧑‍💻 งานที่ 3
เพิ่มตัววัด `marketing_qualified_revenue` (นิยามของฝ่ายการตลาด) ลงใน `METRIC_STORE`
แล้วเขียนฟังก์ชัน `report(metric_name, by=None)` ที่

* คำนวณตัววัดจากทะเบียนกลาง ห้ามเขียนสูตรซ้ำ
* ถ้าระบุ `by` ให้แตกตามมิตินั้น (เช่น `"channel"` หรือ `"store_id"`)
* ถ้าเรียกชื่อตัววัดที่ไม่มีในทะเบียน ให้ raise `KeyError` พร้อมรายชื่อที่มี

In [ ]:
# เขียนโค้ดของคุณที่นี่


> **จุดสำคัญ** ไม่มีที่ใดในโค้ดรายงานที่เขียนสูตรรายได้ซ้ำอีกเลย
> ถ้าฝ่ายบัญชีเปลี่ยนนิยาม แก้ที่ `METRIC_STORE` ที่เดียว ทุกรายงานเปลี่ยนตามทันที
>
> นี่คือแก่นของ **metrics-as-code** — นิยามอยู่ใน version control
> มีเจ้าของ มีประวัติการแก้ไข และ review ได้เหมือนโค้ดทั่วไป

## ส่วนที่ 4 — ทดสอบนิยาม

นิยามที่เป็นโค้ดได้ ย่อมทดสอบได้

### 🧑‍💻 งานที่ 4
เขียนชุดทดสอบอย่างน้อย 4 ข้อสำหรับทะเบียนตัววัด เช่น

1. ตัววัดทุกตัวต้องมี `owner`, `use_for`, `not_for` ครบ
2. `net_revenue_recognized` ต้องน้อยกว่า `gross_revenue` เสมอ
3. ผลรวมของตัววัดเมื่อแตกตามมิติใด ๆ ต้องเท่ากับค่ารวม (additivity)
4. ค่าของตัววัดต้องไม่เปลี่ยนเมื่อเรียงลำดับแถวใหม่ (determinism)

In [ ]:
# เขียนโค้ดของคุณที่นี่


> **ข้อควรระวัง** ข้อ 3 ใช้ได้เฉพาะกับตัววัดแบบ **additive**
> ตัววัดอย่าง *อัตราการคืนสินค้า* หรือ *จำนวนลูกค้าที่ไม่ซ้ำ* เป็น **non-additive**
> ผลรวมของกลุ่มย่อยจะไม่เท่ากับค่ารวม — ต้องเขียนกฎทดสอบคนละแบบ

### 🧑‍💻 งานที่ 5
เพิ่มตัววัด `return_rate` (อัตราการคืนสินค้า = returned ÷ net) เข้าทะเบียน
แล้ว**พิสูจน์ด้วยตัวเลข**ว่าถ้าใช้กฎทดสอบข้อ 3 กับตัววัดนี้ จะไม่ผ่าน
จากนั้นเขียนกฎที่ถูกต้องสำหรับตัววัดแบบ non-additive

In [ ]:
# เขียนโค้ดของคุณที่นี่


## ส่วนที่ 5 — ขอบเขตของ semantic layer

### 🧑‍💻 งานที่ 6 (เขียนเป็นข้อความ)

1. semantic layer **แก้ปัญหาอะไรได้** ในกรณีนี้ — ตอบเป็นข้อ ๆ
2. semantic layer **แก้ปัญหาอะไรไม่ได้** — ยกอย่างน้อย 2 ตัวอย่าง
3. ถ้าองค์กรติดตั้ง semantic layer แล้วแต่ยังมีตัวเลขไม่ตรงกันอยู่
   ให้ตั้งสมมติฐาน 3 ข้อว่าสาเหตุน่าจะเป็นอะไร
4. เขียนนิยามตัววัด `net_revenue_recognized` ในรูปแบบ YAML
   ที่มี: ชื่อ · คำอธิบายภาษาไทย · สูตร · เจ้าของ · มิติที่ใช้แตกได้ · ตัววัดที่ห้ามสับสนด้วย

In [ ]:
# เขียนโค้ดของคุณที่นี่


---
## ✅ เกณฑ์การส่งงาน

| องค์ประกอบ | คะแนน |
|---|:--:|
| งานที่ 1 — คำนวณตัวเลขทั้ง 4 ฝ่ายและช่วงห่างได้ถูกต้อง | 3 |
| งานที่ 2 — จับคู่นิยามกับคำถามที่ตอบได้/ตอบไม่ได้ | 2 |
| งานที่ 3 — สร้าง metric store ที่ไม่มีสูตรซ้ำและมี `by` ใช้งานได้ | 3 |
| งานที่ 4 — ชุดทดสอบครบ 4 ข้อและผ่านทั้งหมด | 3 |
| งานที่ 5 — พิสูจน์ปัญหา non-additive และเสนอกฎที่ถูกต้อง | 2 |
| งานที่ 6 — ขอบเขตของ semantic layer และนิยาม YAML | 3 |
| **รวม** | **16** |

> 💡 ตัวเลขทุกตัวในสมุดเล่มนี้ต้องตรงกับที่แสดงบนสื่อจำลอง `/sims/metric-sprawl`
> ถ้าไม่ตรง แปลว่ามีขั้นตอนใดขั้นตอนหนึ่งผิด — ให้ย้อนกลับไปตรวจก่อนส่ง